In [0]:
from pyspark.sql import functions as F

hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
scotland = (
    hpi
    .filter(F.col("area_code") == "S92000003")
    .select(
        "report_date",
        "year_month",
        F.col("average_price").alias("scotland_avg_price"),
        F.col("sales_volume").alias("scotland_sales_volume")
    )
)

edinburgh = (
    hpi
    .filter(F.col("area_code") == "S12000036")
    .select(
        "year_month",
        F.col("average_price").alias("edinburgh_avg_price"),
        F.col("sales_volume").alias("edinburgh_sales_volume")
    )
)

In [0]:
gold = (
    scotland
    .join(edinburgh, on="year_month", how="inner")
    .withColumn(
        "edinburgh_premium_pct",
        F.round(
            (F.col("edinburgh_avg_price") - F.col("scotland_avg_price"))
            / F.col("scotland_avg_price") * F.lit(100),
            1
        )
    )
    .withColumn(
        "edinburgh_premium_gbp",
        F.col("edinburgh_avg_price") - F.col("scotland_avg_price")
    )
    .select(
        "report_date",
        "year_month",
        "scotland_avg_price",
        "edinburgh_avg_price",
        "edinburgh_premium_gbp",
        "edinburgh_premium_pct",
        "scotland_sales_volume",
        "edinburgh_sales_volume"
    )
    .orderBy("report_date")
)


In [0]:
print(f"Row count: {gold.count()}")
gold.orderBy("report_date").show(5)
gold.orderBy(F.col("report_date").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.edinburgh_vs_scotland_price_comparison")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_edinburgh_vs_scotland_price_comparison/")
)

print("Gold 06 complete.")

In [0]:
# Edinburgh vs Scotland Price Comparison
# Source: silver.uk_hpi only (self-join)
# Grain: monthly
# Join key: year_month
#
# Brings Scotland national and City of Edinburgh onto the same row
# so price gap and relative growth can be analysed directly.
# Edinburgh area code: S12000036
# Scotland national area code: S92000003